# QiBasic CodeGPT — Fine-Tuning on Google Colab (Free GPU)

**How to use this:**
1. Go to https://colab.research.google.com
2. File → Upload notebook → upload this file
3. Runtime → Change runtime type → T4 GPU (free)
4. Run all cells (Runtime → Run all)
5. Download the GGUF file when done
6. Put the GGUF file in your project folder
7. Update Modelfile: change `FROM deepseek-coder` to `FROM ./qibasic_gguf/model.gguf`
8. Run `ollama create qibasic-codegpt -f Modelfile`

In [ ]:
# Cell 1: Install packages (run once)
!pip install unsloth datasets trl -q
print('Packages installed!')

In [ ]:
# Cell 2: Create your training data
# Copy your qibasic_training.jsonl content here
# OR upload the file using the Files panel on the left

training_data = '''
{"instruction": "Write a Hello World program in QBasic", "input": "", "output": "PRINT \\"Hello, World!\\"\\nEND"}
{"instruction": "Write a FOR loop from 1 to 10 in QBasic", "input": "", "output": "FOR i = 1 TO 10\\n    PRINT i\\nNEXT i\\nEND"}
{"instruction": "How do you declare a variable in QBasic?", "input": "", "output": "DIM x AS INTEGER\\nx = 42\\nPRINT x\\nEND"}
'''.strip()

# Save training data to file
with open('qibasic_training.jsonl', 'w') as f:
    f.write(training_data)

# OR if you uploaded the file, skip the above and just verify:
import json
with open('qibasic_training.jsonl') as f:
    lines = [l for l in f if l.strip()]
print(f'Training examples: {len(lines)}')

In [ ]:
# Cell 3: Fine-tune with Unsloth (fast GPU training)
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print('Loading DeepSeek-Coder...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='deepseek-ai/deepseek-coder-1.3b-instruct',
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

PROMPT = """Below is an instruction about QBasic programming. Write a correct QBasic response.

### Instruction:
{instruction}

### Response:
{output}<|endoftext|>"""

dataset = load_dataset('json', data_files='qibasic_training.jsonl', split='train')
dataset = dataset.map(lambda e: {'text': PROMPT.format(instruction=e['instruction'], output=e['output'])})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir='qibasic_finetuned',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        optim='adamw_8bit',
    ),
)

print('Training started...')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 4: Export to GGUF for Ollama
import os
os.makedirs('qibasic_gguf', exist_ok=True)

print('Exporting to GGUF format...')
model.save_pretrained_gguf(
    'qibasic_gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('Done! GGUF file saved to qibasic_gguf/')

In [ ]:
# Cell 5: Download the GGUF file to your PC
import os
from google.colab import files

# Find the GGUF file
for f in os.listdir('qibasic_gguf'):
    if f.endswith('.gguf'):
        print(f'Downloading: {f}')
        files.download(f'qibasic_gguf/{f}')

print()
print('INSTRUCTIONS AFTER DOWNLOAD:')
print('1. Put the downloaded .gguf file in your project folder')
print('2. Rename it to model.gguf')
print('3. Edit Modelfile line 1 to: FROM ./model.gguf')
print('4. Run: ollama create qibasic-codegpt -f Modelfile')
print('5. Run: ollama run qibasic-codegpt')